# Part 1 - Exploratory data analysis of the Porto taxi dataset

This notebook examines `porto/porto.csv` before it is inserted into MySQL. What we find here decides the
cleaning rules and the schema in `part1_import.py`, and screenshots of the results go into the report.

The first cells load the data and work as they are. The numbered sections below are the analysis still to be
written: each lists the questions to answer, with an empty cell to start from.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

CSV_PATH = "porto/porto.csv"

## Loading the data

The `POLYLINE` column makes up almost all of the 1.9 GB file, so `load_trips` reads the CSV in chunks and keeps
only the trip metadata plus the number of GPS points per trip. `load_polyline_sample` gives the actual
coordinates for a sample of the trips.

In [ ]:
def load_trips(csv_path=CSV_PATH, nrows=None, chunksize=100_000):
    """One row per CSV row, with POLYLINE replaced by POINT_COUNT (its number of GPS points)."""
    chunks = []
    reader = pd.read_csv(csv_path, nrows=nrows, chunksize=chunksize,
                         dtype={"ORIGIN_CALL": "Int64", "ORIGIN_STAND": "Int64"})
    for chunk in reader:
        polyline = chunk.pop("POLYLINE")
        # "[]" has no points, "[[lon,lat]]" has one, and every further point adds one "],["
        chunk["POINT_COUNT"] = (polyline.str.count(r"\],\[") + 1).where(polyline != "[]", 0)
        chunk["START_TIME"] = pd.to_datetime(chunk["TIMESTAMP"], unit="s")  # UTC
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


def load_polyline_sample(csv_path=CSV_PATH, every=50):
    """The parsed POLYLINE ([[longitude, latitude], ...]) of every n-th trip, indexed by TRIP_ID."""
    sample = pd.read_csv(csv_path, usecols=["TRIP_ID", "POLYLINE"],
                         skiprows=lambda row: row > 0 and row % every != 0)
    return sample.set_index("TRIP_ID")["POLYLINE"].map(json.loads)

In [ ]:
trips = load_trips()
trips.info()
trips.head()

## 1. Structure

- How many rows and columns are there, and which types do the columns have?
- Which columns contain missing values, and how many?
- Does the file match the column descriptions in the assignment sheet? (The sheet says `DAYTYPE`, the file says `DAY_TYPE`.)

In [ ]:
# TODO

## 2. TRIP_ID

- Is `TRIP_ID` unique, as the assignment sheet says? How many ids appear more than once?
- Are the rows sharing an id identical copies or different trips? How do they differ?
- How is the id built up? Does that explain the duplicates?

In [ ]:
# TODO

## 3. CALL_TYPE, ORIGIN_CALL and ORIGIN_STAND

- How are the trips distributed over the three call types?
- Is `ORIGIN_CALL` set exactly when `CALL_TYPE` is A, and `ORIGIN_STAND` exactly when it is B? How many exceptions are there?
- How many distinct clients and taxi stands are there, and which are used the most?

In [ ]:
# TODO

## 4. TAXI_ID

- How many taxis are there?
- How are the trips distributed over the taxis?

In [ ]:
# TODO

## 5. TIMESTAMP

- Which period does the data cover?
- How are the trips distributed over months, weekdays and hours of the day?
- The timestamps are in UTC and Porto is on UTC+1 in summer. Does that matter for our queries?

In [ ]:
# TODO

## 6. DAY_TYPE

- How are the trips distributed over the three day types?
- Does the column tell us anything? Compare with a known holiday, e.g. 2013-12-25.

In [ ]:
# TODO

## 7. MISSING_DATA and trajectory length

- How many trips have `MISSING_DATA` set, and what do those trips look like?
- How is `POINT_COUNT` distributed? What are the shortest and the longest trips?
- How many trips have an empty `POLYLINE`, and how many have fewer than 3 points? Do they differ from the other trips (call type, taxi, time)?
- What do missing points mean for the trip duration we derive as (points - 1) x 15 s?

In [ ]:
# TODO

## 8. GPS points

- Within which longitudes and latitudes do the points lie? Plot them. (Start from `polylines = load_polyline_sample()`.)
- How many points are far away from Porto?
- How far apart are consecutive points? How many jumps are impossible for a taxi in 15 seconds?

In [ ]:
# TODO

## 9. Conclusions

- For each data quality issue found above: how many rows it affects, and how `part1_import.py` handles it.
- Which cleaning rules or parts of the schema should change because of what we found?